# TDD Cycle Dashboard

Visualizes results across multiple TDD cycle runs: Build, Test, Coverage, and Code Metrics.

In [1]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path("..")


def load_build_results():
    """Load all build-summary.json files into a DataFrame."""
    rows = []
    base = REPO_ROOT / "BuildResults"
    if not base.exists():
        return pd.DataFrame()
    for ts_dir in sorted(base.iterdir()):
        summary = ts_dir / "build-summary.json"
        if summary.exists():
            data = json.loads(summary.read_text())
            rows.append({
                "timestamp": ts_dir.name,
                "status": data.get("status", "unknown"),
                "totalErrors": data.get("totalErrors", 0),
                "totalWarnings": data.get("totalWarnings", 0),
                "projectCount": len(data.get("projects", [])),
                "failedProjects": sum(1 for p in data.get("projects", []) if p.get("status") == "failure"),
            })
    return pd.DataFrame(rows)


def load_test_results():
    """Load all test-summary.json files into a DataFrame."""
    rows = []
    base = REPO_ROOT / "TestResults"
    if not base.exists():
        return pd.DataFrame()
    for ts_dir in sorted(base.iterdir()):
        summary = ts_dir / "test-summary.json"
        if summary.exists():
            data = json.loads(summary.read_text())
            rows.append({
                "timestamp": ts_dir.name,
                "status": data.get("status", "unknown"),
                "totalTests": data.get("totalTests", 0),
                "totalPassed": data.get("totalPassed", 0),
                "totalFailed": data.get("totalFailed", 0),
                "totalSkipped": data.get("totalSkipped", 0),
            })
    return pd.DataFrame(rows)


def load_coverage_results():
    """Load coverage from Combined/Cobertura.xml in each TestResults timestamp."""
    rows = []
    base = REPO_ROOT / "TestResults"
    if not base.exists():
        return pd.DataFrame()
    for ts_dir in sorted(base.iterdir()):
        cob = ts_dir / "Coverage" / "Combined" / "Cobertura.xml"
        if cob.exists():
            try:
                tree = ET.parse(cob)
                root = tree.getroot()
                rows.append({
                    "timestamp": ts_dir.name,
                    "lineRate": float(root.get("line-rate", 0)),
                    "branchRate": float(root.get("branch-rate", 0)),
                    "linesCovered": int(root.get("lines-covered", 0)),
                    "linesValid": int(root.get("lines-valid", 0)),
                    "branchesCovered": int(root.get("branches-covered", 0)),
                    "branchesValid": int(root.get("branches-valid", 0)),
                    "complexity": int(root.get("complexity", 0)),
                })
            except ET.ParseError:
                pass
    return pd.DataFrame(rows)


def load_metrics_results():
    """Load all metrics-summary.json files into a list of (storyId, timestamp, data) tuples."""
    rows = []
    base = REPO_ROOT / "MetricsResults"
    if not base.exists():
        return pd.DataFrame()
    for story_dir in sorted(base.iterdir()):
        if not story_dir.is_dir():
            continue
        for ts_dir in sorted(story_dir.iterdir()):
            summary = ts_dir / "metrics-summary.json"
            if summary.exists():
                data = json.loads(summary.read_text())
                for proj in data.get("projects", []):
                    for t in proj.get("types", []):
                        rows.append({
                            "storyId": story_dir.name,
                            "timestamp": ts_dir.name,
                            "project": proj["name"],
                            "type": t["name"],
                            "maintainabilityIndex": t["maintainabilityIndex"]["value"],
                            "miFlag": t["maintainabilityIndex"]["flag"],
                            "cyclomaticComplexity": t["cyclomaticComplexity"]["value"],
                            "ccFlag": t["cyclomaticComplexity"]["flag"],
                            "classCoupling": t["classCoupling"]["value"],
                            "couplingFlag": t["classCoupling"]["flag"],
                            "depthOfInheritance": t["depthOfInheritance"]["value"],
                            "ditFlag": t["depthOfInheritance"]["flag"],
                            "sourceLines": t["sourceLines"],
                            "executableLines": t["executableLines"],
                        })
    return pd.DataFrame(rows)


# Load all data
df_build = load_build_results()
df_test = load_test_results()
df_coverage = load_coverage_results()
df_metrics = load_metrics_results()

print(f"Build runs: {len(df_build)}")
print(f"Test runs: {len(df_test)}")
print(f"Coverage runs: {len(df_coverage)}")
print(f"Metrics entries: {len(df_metrics)}")

Build runs: 8
Test runs: 2
Coverage runs: 2
Metrics entries: 49


## Build Results Over Time

In [2]:
if not df_build.empty:
    # Build status bar chart
    color_map = {"success": "#2ecc71", "failure": "#e74c3c"}
    fig = px.bar(
        df_build, x="timestamp", y="totalErrors",
        color="status", color_discrete_map=color_map,
        title="Build Status & Errors Over Time",
        labels={"totalErrors": "Total Errors", "timestamp": "Run"},
    )
    fig.add_scatter(
        x=df_build["timestamp"], y=df_build["totalWarnings"],
        name="Warnings", mode="lines+markers",
        line=dict(color="#f39c12", dash="dot"),
    )
    fig.update_layout(xaxis_tickangle=-45, height=400)
    fig.show()
else:
    print("No build results found.")

## Test Results Over Time

In [3]:
if not df_test.empty:
    fig = go.Figure()
    fig.add_bar(x=df_test["timestamp"], y=df_test["totalPassed"], name="Passed", marker_color="#2ecc71")
    fig.add_bar(x=df_test["timestamp"], y=df_test["totalFailed"], name="Failed", marker_color="#e74c3c")
    fig.add_bar(x=df_test["timestamp"], y=df_test["totalSkipped"], name="Skipped", marker_color="#95a5a6")
    fig.update_layout(
        barmode="stack",
        title="Test Results Over Time",
        xaxis_title="Run", yaxis_title="Test Count",
        xaxis_tickangle=-45, height=400,
    )
    fig.show()

    # Pass rate trend
    df_test_rate = df_test.copy()
    df_test_rate["passRate"] = df_test_rate.apply(
        lambda r: (r["totalPassed"] / r["totalTests"] * 100) if r["totalTests"] > 0 else 0, axis=1
    )
    fig2 = px.line(
        df_test_rate, x="timestamp", y="passRate",
        title="Test Pass Rate (%)", markers=True,
        labels={"passRate": "Pass Rate (%)", "timestamp": "Run"},
    )
    fig2.update_layout(yaxis_range=[0, 105], xaxis_tickangle=-45, height=350)
    fig2.show()
else:
    print("No test results found.")

## Code Coverage Over Time

In [4]:
if not df_coverage.empty:
    fig = go.Figure()
    fig.add_scatter(
        x=df_coverage["timestamp"], y=df_coverage["lineRate"] * 100,
        name="Line Coverage %", mode="lines+markers", line=dict(color="#3498db"),
    )
    fig.add_scatter(
        x=df_coverage["timestamp"], y=df_coverage["branchRate"] * 100,
        name="Branch Coverage %", mode="lines+markers", line=dict(color="#9b59b6"),
    )
    fig.update_layout(
        title="Code Coverage Over Time",
        yaxis_title="Coverage %", xaxis_title="Run",
        yaxis_range=[0, 105], xaxis_tickangle=-45, height=400,
    )
    fig.show()

    # Lines covered vs valid
    fig2 = go.Figure()
    fig2.add_bar(x=df_coverage["timestamp"], y=df_coverage["linesCovered"], name="Lines Covered", marker_color="#2ecc71")
    fig2.add_bar(x=df_coverage["timestamp"], y=df_coverage["linesValid"] - df_coverage["linesCovered"], name="Lines Not Covered", marker_color="#e74c3c")
    fig2.update_layout(
        barmode="stack", title="Lines Covered vs Total",
        xaxis_title="Run", yaxis_title="Lines",
        xaxis_tickangle=-45, height=350,
    )
    fig2.show()
else:
    print("No coverage data found.")

## Code Metrics Over Time (per Story)

In [5]:
if not df_metrics.empty:
    # Maintainability Index by type across runs
    fig = px.bar(
        df_metrics, x="type", y="maintainabilityIndex", color="timestamp",
        barmode="group",
        title="Maintainability Index by Type (per run)",
        labels={"maintainabilityIndex": "MI", "type": "Type"},
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.add_hline(y=20, line_dash="dash", line_color="red", annotation_text="RED threshold")
    fig.add_hline(y=10, line_dash="dash", line_color="orange", annotation_text="YELLOW threshold")
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()

    # Cyclomatic Complexity vs Class Coupling scatter
    fig2 = px.scatter(
        df_metrics, x="cyclomaticComplexity", y="classCoupling",
        color="miFlag", symbol="timestamp",
        size="sourceLines", hover_name="type",
        title="Cyclomatic Complexity vs Class Coupling",
        labels={"cyclomaticComplexity": "Cyclomatic Complexity", "classCoupling": "Class Coupling"},
        color_discrete_map={"GREEN": "#2ecc71", "YELLOW": "#f39c12", "RED": "#e74c3c"},
    )
    fig2.add_vline(x=10, line_dash="dash", line_color="orange", annotation_text="CC threshold")
    fig2.add_hline(y=9, line_dash="dash", line_color="orange", annotation_text="Coupling threshold")
    fig2.update_layout(height=500)
    fig2.show()

    # Flag distribution per run
    flag_counts = df_metrics.groupby(["timestamp", "miFlag"]).size().reset_index(name="count")
    fig3 = px.bar(
        flag_counts, x="timestamp", y="count", color="miFlag",
        color_discrete_map={"GREEN": "#2ecc71", "YELLOW": "#f39c12", "RED": "#e74c3c"},
        title="Maintainability Index Flag Distribution per Run",
        labels={"count": "Type Count", "miFlag": "Flag"},
    )
    fig3.update_layout(xaxis_tickangle=-45, height=400)
    fig3.show()
else:
    print("No metrics data found.")

## Metrics Before/After Comparison

In [6]:
if not df_metrics.empty:
    stories = df_metrics["storyId"].unique()
    for story in stories:
        story_df = df_metrics[df_metrics["storyId"] == story].copy()
        timestamps = sorted(story_df["timestamp"].unique())
        if len(timestamps) < 2:
            print(f"Story {story}: only 1 metrics run, skipping before/after comparison.")
            continue

        before_ts, after_ts = timestamps[0], timestamps[-1]

        # Use project+type as unique key to avoid duplicates (e.g., DependencyInjection in multiple projects)
        story_df["key"] = story_df["project"].str.split(".").str[-1] + "." + story_df["type"]

        before = story_df[story_df["timestamp"] == before_ts].drop_duplicates("key").set_index("key")
        after = story_df[story_df["timestamp"] == after_ts].drop_duplicates("key").set_index("key")
        common_keys = sorted(before.index.intersection(after.index))

        if not common_keys:
            continue

        comparison = pd.DataFrame({
            "Type": [before.loc[k, "type"] for k in common_keys],
            "Project": [before.loc[k, "project"].split(".")[-1] for k in common_keys],
            "MI Before": [before.loc[k, "maintainabilityIndex"] for k in common_keys],
            "MI After": [after.loc[k, "maintainabilityIndex"] for k in common_keys],
            "CC Before": [before.loc[k, "cyclomaticComplexity"] for k in common_keys],
            "CC After": [after.loc[k, "cyclomaticComplexity"] for k in common_keys],
            "Coupling Before": [before.loc[k, "classCoupling"] for k in common_keys],
            "Coupling After": [after.loc[k, "classCoupling"] for k in common_keys],
        })
        comparison["MI Delta"] = comparison["MI After"] - comparison["MI Before"]
        comparison["CC Delta"] = comparison["CC After"] - comparison["CC Before"]
        comparison["Coupling Delta"] = comparison["Coupling After"] - comparison["Coupling Before"]

        print(f"\n=== Story: {story} ===")
        print(f"Before: {before_ts} | After: {after_ts}")
        display(comparison)

        # Grouped bar: MI before vs after
        labels = comparison["Project"] + "." + comparison["Type"]
        fig = go.Figure()
        fig.add_bar(x=labels, y=comparison["MI Before"], name=f"Before ({before_ts})", marker_color="#e74c3c")
        fig.add_bar(x=labels, y=comparison["MI After"], name=f"After ({after_ts})", marker_color="#2ecc71")
        fig.update_layout(
            barmode="group",
            title=f"Maintainability Index: Before vs After — {story}",
            yaxis_title="MI", xaxis_tickangle=-45, height=400,
        )
        fig.add_hline(y=20, line_dash="dash", line_color="red")
        fig.show()
else:
    print("No metrics data for comparison.")


=== Story: CPD-LC-001-001 ===
Before: 2026-04-18_13-05-20 | After: 2026-04-18_13-09-45


,Type,Project,MI Before,MI After,CC Before,CC After,Coupling Before,Coupling After,MI Delta,CC Delta,Coupling Delta
0,DependencyInjection,Application,75,75,1,1,2,2,0,0,0
1,ILearningComponentService,Application,79,79,1,1,2,2,0,0,0
2,ILearningSpaceListService,Application,75,75,2,2,3,3,0,0,0
3,LearningComponentService,Application,68,68,3,3,5,5,0,0,0
4,LearningSpaceListService,Application,67,67,3,3,5,5,0,0,0
5,DependencyInjection,DependencyInjection,74,74,1,1,4,4,0,0,0
6,ILearningComponentRepository,Domain,79,79,1,1,2,2,0,0,0
7,ILearningSpaceListRepository,Domain,76,76,2,2,3,3,0,0,0
8,LearningComponent,Domain,55,55,8,8,2,2,0,0,0
9,LearningSpace,Domain,64,64,1,1,0,0,0,0,0


## Combined Dashboard Summary

In [7]:
# Summary cards
print("=" * 60)
print("  TDD CYCLE DASHBOARD SUMMARY")
print("=" * 60)

if not df_build.empty:
    latest_build = df_build.iloc[-1]
    print(f"  Latest Build: {latest_build['status'].upper()} ({latest_build['timestamp']})")
    print(f"  Total Build Runs: {len(df_build)} | Failures: {(df_build['status'] == 'failure').sum()}")

if not df_test.empty:
    latest_test = df_test.iloc[-1]
    rate = (latest_test['totalPassed'] / latest_test['totalTests'] * 100) if latest_test['totalTests'] > 0 else 0
    print(f"  Latest Tests: {latest_test['totalPassed']}/{latest_test['totalTests']} passed ({rate:.0f}%)")
    print(f"  Total Test Runs: {len(df_test)}")

if not df_coverage.empty:
    latest_cov = df_coverage.iloc[-1]
    print(f"  Latest Coverage: Line={latest_cov['lineRate']*100:.1f}% Branch={latest_cov['branchRate']*100:.1f}%")

if not df_metrics.empty:
    latest_ts = df_metrics["timestamp"].max()
    latest_m = df_metrics[df_metrics["timestamp"] == latest_ts]
    print(f"  Latest Metrics ({latest_ts}):")
    print(f"    Avg MI: {latest_m['maintainabilityIndex'].mean():.0f}")
    print(f"    Max CC: {latest_m['cyclomaticComplexity'].max()}")
    print(f"    Max Coupling: {latest_m['classCoupling'].max()}")
    green = (latest_m['miFlag'] == 'GREEN').sum()
    yellow = (latest_m['miFlag'] == 'YELLOW').sum()
    red = (latest_m['miFlag'] == 'RED').sum()
    print(f"    MI Flags: {green} GREEN, {yellow} YELLOW, {red} RED")

print("=" * 60)

  TDD CYCLE DASHBOARD SUMMARY
  Latest Build: SUCCESS (2026-04-18_13-09-06)
  Total Build Runs: 8 | Failures: 6
  Latest Tests: 24/24 passed (100%)
  Total Test Runs: 2
  Latest Coverage: Line=53.9% Branch=90.0%
  Latest Metrics (2026-04-18_13-09-45):
    Avg MI: 74
    Max CC: 8
    Max Coupling: 17
    MI Flags: 25 GREEN, 0 YELLOW, 0 RED
